In [2]:
# ============================================================
# 0. INSTALL
# ============================================================
!pip install lpips pytorch-msssim ipywidgets --quiet

# ============================================================
# 1. IMPORTS
# ============================================================
import os
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from pytorch_msssim import ssim as ssim_fn
import lpips
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ============================================================
# 2. PATHS
# ============================================================
gt_dir = '/kaggle/input/datasets/dd1vya/semcon/dataset/train/GT'
lr_dir = '/kaggle/input/datasets/dd1vya/semcon/dataset/train/NoisyLR'
test_dir = '/kaggle/input/datasets/dd1vya/semcon/dataset/test/NoisyLR'

# ============================================================
# 3. SEEDING
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

# ============================================================
# 4. DEVICE
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# ============================================================
# 5. TRAIN/VAL SPLIT
# ============================================================
all_files = sorted(os.listdir(gt_dir))
random.shuffle(all_files)

n_val = int(0.1 * len(all_files))
val_files = set(all_files[:n_val])
train_files = [f for f in all_files if f not in val_files]
val_files = list(val_files)

print(f"Train: {len(train_files)}, Val: {len(val_files)}")

# ============================================================
# 6. DATASET CLASS
# ============================================================
class RestorationDataset(Dataset):
    def __init__(self, gt_dir, lr_dir, file_list, patch_size_hr=128, train=True, p_synthetic=0.15):
        self.gt_dir = gt_dir
        self.lr_dir = lr_dir
        self.files = file_list
        self.patch_size_hr = patch_size_hr
        self.train = train
        self.p_synthetic = p_synthetic 

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        gt = np.load(os.path.join(self.gt_dir, fname)).astype(np.float32)
        lr = np.load(os.path.join(self.lr_dir, fname)).astype(np.float32)

        if self.train:
            gt, lr = self._aligned_crop(gt, lr, self.patch_size_hr)
            gt, lr = self._augment(gt, lr)
            if random.random() < self.p_synthetic:
                lr = self._synthesize_degradation(gt)

        gt = np.clip(gt, 0.0, 1.0)

        gt_t = torch.from_numpy(gt).unsqueeze(0)
        lr_t = torch.from_numpy(lr).unsqueeze(0)
        return lr_t, gt_t

    def _aligned_crop(self, gt, lr, patch_hr):
        patch_lr = patch_hr // 2
        h_lr, w_lr = lr.shape
        x = random.randint(0, w_lr - patch_lr)
        y = random.randint(0, h_lr - patch_lr)
        lr_crop = lr[y:y+patch_lr, x:x+patch_lr]
        gt_crop = gt[y*2:y*2+patch_hr, x*2:x*2+patch_hr]
        return gt_crop, lr_crop

    def _augment(self, gt, lr):
        if random.random() < 0.5:
            gt, lr = np.fliplr(gt).copy(), np.fliplr(lr).copy()
        k = random.randint(0, 3)
        gt, lr = np.rot90(gt, k).copy(), np.rot90(lr, k).copy()
        return gt, lr

    def _synthesize_degradation(self, gt_patch):
        img = gt_patch.astype(np.float32).copy()
        H, W = img.shape
        lr = img.reshape(H // 2, 2, W // 2, 2).mean(axis=(1, 3))
        speckle_std = random.uniform(0.05, 0.25)
        noise = np.random.randn(*lr.shape).astype(np.float32)
        lr = lr + lr * noise * speckle_std
        gauss_std = random.uniform(0.0, 0.03)

        if gauss_std > 0:
            lr = lr + np.random.randn(*lr.shape).astype(np.float32) * gauss_std

        return lr.astype(np.float32)

  

# ============================================================
# 7. DATALOADER INSTANCES
# ============================================================
train_dataset = RestorationDataset(gt_dir, lr_dir, train_files, patch_size_hr=128, train=True)
val_dataset   = RestorationDataset(gt_dir, lr_dir, val_files, patch_size_hr=128, train=False)

train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True,
    num_workers=2, worker_init_fn=seed_worker, generator=g
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=2, worker_init_fn=seed_worker, generator=g
)

# ============================================================
# 8. ARCHITECTURE
# ============================================================
class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class SimplifiedChannelAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(c, c, 1)

    def forward(self, x):
        return x * self.conv(self.pool(x))

class NAFBlock(nn.Module):
    def __init__(self, c, expand=2):
        super().__init__()
        dw_c = c * expand
        self.norm1 = nn.GroupNorm(1, c)
        self.conv1 = nn.Conv2d(c, dw_c, 1)
        self.dwconv = nn.Conv2d(dw_c, dw_c, 3, padding=1, groups=dw_c)
        self.sg = SimpleGate()
        self.sca = SimplifiedChannelAttention(dw_c // 2)
        self.conv2 = nn.Conv2d(dw_c // 2, c, 1)

        self.norm2 = nn.GroupNorm(1, c)
        self.conv3 = nn.Conv2d(c, dw_c, 1)
        self.sg2 = SimpleGate()
        self.conv4 = nn.Conv2d(dw_c // 2, c, 1)

        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, x):
        y = self.norm1(x)
        y = self.conv1(y)
        y = self.dwconv(y)
        y = self.sg(y)
        y = self.sca(y)
        y = self.conv2(y)
        x = x + y * self.beta

        y = self.norm2(x)
        y = self.conv3(y)
        y = self.sg2(y)
        y = self.conv4(y)
        x = x + y * self.gamma
        return x

# ------------------------------------------------------------
# ICNR INITIALIZATION
# ------------------------------------------------------------
def icnr_init(weight, scale=2, init=nn.init.kaiming_normal_):
    """
    ICNR initialization keeps the sub-pixel convolution
    channels synchronized at initialization, reducing
    checkerboard artifacts from PixelShuffle.
    """
    out_c, in_c, kh, kw = weight.shape
    base_c = out_c // (scale ** 2)
    base = torch.zeros(base_c, in_c, kh, kw)
    init(base)
    kernel = base.repeat_interleave(scale ** 2, dim=0)
    weight.data.copy_(kernel)

class NAFNetSR(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, width=32, enc_blocks=(2,2,4), mid_blocks=4, dec_blocks=(2,2,2), scale=2):
        super().__init__()
        self.scale = scale  
        self.intro = nn.Conv2d(in_ch, width, 3, padding=1)

        self.encoders = nn.ModuleList()
        self.downs = nn.ModuleList()
        c = width
        for n in enc_blocks:
            self.encoders.append(nn.Sequential(*[NAFBlock(c) for _ in range(n)]))
            self.downs.append(nn.Conv2d(c, c*2, 2, stride=2))
            c *= 2

        self.middle = nn.Sequential(*[NAFBlock(c) for _ in range(mid_blocks)])

        self.ups = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for n in dec_blocks:
            self.ups.append(nn.Sequential(nn.Conv2d(c, c*2, 1), nn.PixelShuffle(2)))
            c //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(c) for _ in range(n)]))

        self.sr_head = nn.Sequential(
            nn.Conv2d(c, c * (scale**2), 3, padding=1),
            nn.PixelShuffle(scale)
        )
        icnr_init(self.sr_head[0].weight, scale=scale)

        self.outro = nn.Conv2d(c, out_ch, 3, padding=1)

    def forward(self, x):
        lr_input = x  # keep the raw input around for the global residual

        x = self.intro(x)
        skips = []
        for enc, down in zip(self.encoders, self.downs):
            x = enc(x)
            skips.append(x)
            x = down(x)

        x = self.middle(x)

        for up, dec, skip in zip(self.ups, self.decoders, reversed(skips)):
            x = up(x)
            x = x + skip
            x = dec(x)

        x = self.sr_head(x)
        x = self.outro(x)

        # global residual — predict a correction on top of an upsampled LR
        # base instead of reconstructing the whole image from scratch.
        base = F.interpolate(lr_input, scale_factor=self.scale, mode='bicubic', align_corners=False)
        x = x + base
        return x

# ============================================================
# 9. METRICS 
# ============================================================
def compute_psnr(pred, gt):
    pred_np = pred.clamp(0, 1).squeeze().detach().cpu().numpy()
    gt_np = gt.clamp(0, 1).squeeze().detach().cpu().numpy()
    return psnr(gt_np, pred_np, data_range=1.0)

def compute_ssim(pred, gt):
    pred_np = pred.clamp(0, 1).squeeze().detach().cpu().numpy()
    gt_np = gt.clamp(0, 1).squeeze().detach().cpu().numpy()
    return ssim(gt_np, pred_np, data_range=1.0)

def compute_lpips_metric(pred, gt, lpips_fn):
    pred_c = pred.clamp(0, 1).repeat(1, 3, 1, 1) * 2 - 1
    gt_c = gt.clamp(0, 1).repeat(1, 3, 1, 1) * 2 - 1
    with torch.no_grad():
        return lpips_fn(pred_c, gt_c).item()

# ============================================================
# 10. COMPOSITE LOSS 
# ============================================================
class CompositeLoss(nn.Module):
    def __init__(self, w_l1=0.6, w_ssim=0.2, w_lpips=0.2, device='cuda'):
        super().__init__()
        self.w_l1 = w_l1
        self.w_ssim = w_ssim
        self.w_lpips = w_lpips

        self.lpips_fn = lpips.LPIPS(net='alex').to(device)
        for p in self.lpips_fn.parameters():
            p.requires_grad = False

    def forward(self, pred, target):
        # pred is RAW (no clamp) — L1 below gets full gradient on all pixels
        l1 = F.l1_loss(pred, target)

        # SSIM/LPIPS: clamp only for these sub-terms
        pred_c = pred.clamp(0, 1)
        target_c = target.clamp(0, 1)

        ssim_val = ssim_fn(pred_c, target_c, data_range=1.0, size_average=True)
        ssim_loss = 1.0 - ssim_val

        pred_3ch = pred_c.repeat(1, 3, 1, 1) * 2 - 1
        target_3ch = target_c.repeat(1, 3, 1, 1) * 2 - 1
        lpips_val = self.lpips_fn(pred_3ch, target_3ch).mean()

        total = self.w_l1 * l1 + self.w_ssim * ssim_loss + self.w_lpips * lpips_val

        return total, {
            'l1': l1.item(),
            'ssim_loss': ssim_loss.item(),
            'lpips': lpips_val.item(),
            'total': total.item()
        }

# ============================================================
# 11. TRAINING LOOP (
# ============================================================
criterion = CompositeLoss(w_l1=0.6, w_ssim=0.2, w_lpips=0.2, device=device)

num_epochs = 63
checkpoint_path = '/kaggle/working/best_model_composite.pt'

model = NAFNetSR().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)

best_val_psnr = -1.0

for epoch in range(num_epochs):
    start = time.time()

    model.train()
    train_loss_total, l1_total, ssim_total, lpips_total = 0.0, 0.0, 0.0, 0.0

    for lr_batch, gt_batch in train_loader:
        lr_batch, gt_batch = lr_batch.to(device), gt_batch.to(device)

        opt.zero_grad()
        pred = model(lr_batch)
        loss, components = criterion(pred, gt_batch)
        loss.backward()
        opt.step()

        bs = lr_batch.size(0)
        train_loss_total += loss.item() * bs
        l1_total += components['l1'] * bs
        ssim_total += components['ssim_loss'] * bs
        lpips_total += components['lpips'] * bs

    n = len(train_dataset)
    train_loss_avg = train_loss_total / n
    scheduler.step()

    model.eval()
    val_psnr_total = 0.0
    val_ssim_total = 0.0
    with torch.no_grad():
        for lr_val, gt_val in val_loader:
            lr_val, gt_val = lr_val.to(device), gt_val.to(device)
            pred_val = model(lr_val)
            val_psnr_total += compute_psnr(pred_val, gt_val)
            val_ssim_total += compute_ssim(pred_val, gt_val)

    val_psnr_avg = val_psnr_total / len(val_dataset)
    val_ssim_avg = val_ssim_total / len(val_dataset)

    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss_avg:.4f} "
          f"(l1={l1_total/n:.4f} ssim_l={ssim_total/n:.4f} lpips={lpips_total/n:.4f}) | "
          f"val_PSNR={val_psnr_avg:.3f} dB | val_SSIM={val_ssim_avg:.4f} | {elapsed:.1f}s")

    if val_psnr_avg > best_val_psnr:
        best_val_psnr = val_psnr_avg
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_psnr': float(val_psnr_avg),
            'val_ssim': float(val_ssim_avg),
        }, checkpoint_path)
        print(f"  -> new best (PSNR={val_psnr_avg:.3f}), checkpoint saved")

print(f"\nTraining done. Best val PSNR: {best_val_psnr:.3f} dB, saved to {checkpoint_path}")

# ============================================================
# 12. FINAL EVALUATION
# ============================================================
def load_checkpoint(path):
    m = NAFNetSR().to(device)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    return m

def evaluate_checkpoint(path, lpips_fn):
    m = load_checkpoint(path)
    psnr_total = ssim_total = lpips_total = 0.0
    with torch.no_grad():
        for lr_val, gt_val in val_loader:
            lr_val, gt_val = lr_val.to(device), gt_val.to(device)
            pred_val = m(lr_val)
            psnr_total += compute_psnr(pred_val, gt_val)
            ssim_total += compute_ssim(pred_val, gt_val)
            lpips_total += compute_lpips_metric(pred_val, gt_val, lpips_fn)
    n = len(val_dataset)
    return psnr_total / n, ssim_total / n, lpips_total / n

lpips_eval_fn = lpips.LPIPS(net='alex').to(device)
p, s, l = evaluate_checkpoint(checkpoint_path, lpips_eval_fn)
print(f"\nFull composite (0.6/0.2/0.2)  PSNR={p:.3f}  SSIM={s:.4f}  LPIPS={l:.4f}  (lower LPIPS = better)")

Using device: cuda
Train: 2880, Val: 320
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 192MB/s]  


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Epoch 1/63 | train_loss=0.1702 (l1=0.0500 ssim_l=0.3832 lpips=0.3178) | val_PSNR=26.024 dB | val_SSIM=0.6956 | 44.8s
  -> new best (PSNR=26.024), checkpoint saved
Epoch 2/63 | train_loss=0.1295 (l1=0.0372 ssim_l=0.2976 lpips=0.2380) | val_PSNR=26.852 dB | val_SSIM=0.7160 | 24.2s
  -> new best (PSNR=26.852), checkpoint saved
Epoch 3/63 | train_loss=0.1204 (l1=0.0358 ssim_l=0.2853 lpips=0.2093) | val_PSNR=26.907 dB | val_SSIM=0.7171 | 24.4s
  -> new best (PSNR=26.907), checkpoint saved
Epoch 4/63 | train_loss=0.1155 (l1=0.0353 ssim_l=0.2781 lpips=0.1931) | val_PSNR=27.137 dB | val_SSIM=0.7259 | 25.1s
  -> new best (PSNR=27.137), checkpoint saved
Epoch 5/63 | train_loss=0.1138 (l1=0.0349 ssim_l=0.2747 lpips=0.1895) | val_PSNR=26.845 dB | val_SSIM=0.7254 | 24.6s
Epoch 6/63 | train_loss=0.1118 (l1=0.0348 ssim_l=0.2725 lpips=0.1820) | val_PSNR=27.129 dB | val_SSIM=0.7349 | 24.5s
Epoch 7/63 | train_loss=0.